# Logit Lens + RSA — Mistral-7B-Instruct-v0.2 (Multilingual)

This notebook loads pre-computed outputs from `modal_bloom.py` and produces:
- **Figure 2b / Figure 3b** — GT rank and top-1 confidence by layer
- **Figure 4** — RSA cross-lingual correlation by layer
- **Figure 6** — Side-by-side comparison: BLOOM-560M vs Mistral-7B-Instruct-v0.2

**No model inference happens here.** Run `modal run modal_bloom.py` first to generate:
- `mistral_hidden_states.json`
- `mistral_logit_lens_results.json`

Mistral-7B-Instruct-v0.2 specs: **32 transformer layers**, **d_model = 4096**

In [ ]:
# Cell 1 — Load pre-computed Modal outputs
import json
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("figures", exist_ok=True)

CATEGORIES = ["math", "factual", "reasoning"]
LANGS = ["English", "Spanish", "Basque"]
N_LAYERS = 32    # Mistral-7B: 32 transformer layers
D_MODEL  = 4096  # Mistral-7B: hidden size

# ── Load hidden states ────────────────────────────────────────────────────────
with open("mistral_hidden_states.json") as f:
    raw_hs = json.load(f)

# rsa_vectors[lang][cat] = np.array of shape (n_questions, n_layers, d_model)
rsa_vectors = {lang: {cat: [] for cat in CATEGORIES} for lang in LANGS}
for entry in raw_hs:
    rsa_vectors[entry["language"]][entry["category"]].append(entry["hidden_states"])

for lang in LANGS:
    for cat in CATEGORIES:
        rsa_vectors[lang][cat] = np.array(rsa_vectors[lang][cat])

sample_shape = rsa_vectors["English"]["math"].shape
print(f"rsa_vectors['English']['math'].shape = {sample_shape}")
assert sample_shape[1] == N_LAYERS, f"Expected {N_LAYERS} layers, got {sample_shape[1]}"
assert sample_shape[2] == D_MODEL,  f"Expected d_model={D_MODEL}, got {sample_shape[2]}"

# ── Load logit lens results ───────────────────────────────────────────────────
with open("mistral_logit_lens_results.json") as f:
    raw_ll = json.load(f)

results = {lang: {cat: {"ranks": [], "top1": []} for cat in CATEGORIES} for lang in LANGS}
for entry in raw_ll:
    results[entry["language"]][entry["category"]]["ranks"].append(entry["ranks"])
    results[entry["language"]][entry["category"]]["top1"].append(entry["top1_probs"])

for lang in LANGS:
    for cat in CATEGORIES:
        results[lang][cat]["ranks"] = np.array(results[lang][cat]["ranks"])
        results[lang][cat]["top1"]  = np.array(results[lang][cat]["top1"])

print(f"Logit lens ranks shape (English/math): {results['English']['math']['ranks'].shape}")
print("Data loaded successfully.")

In [ ]:
# Cell 2 — Figure 2b + 3b: GT rank and top-1 confidence by layer

layers = list(range(1, N_LAYERS + 1))
lang_styles = {
    "English": ("steelblue",   "o"),
    "Spanish": ("darkorange",  "s"),
    "Basque":  ("green",       "^"),
}

# --- Figure 2b: GT rank by layer ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, cat in zip(axes, CATEGORIES):
    for lang, (color, marker) in lang_styles.items():
        rm = results[lang][cat]["ranks"]
        ax.plot(layers, rm.mean(axis=0), marker=marker, color=color,
                label=lang, linewidth=1.8, markersize=4)
    ax.set_title(cat.capitalize(), fontsize=11)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean rank of GT token  (lower = better)")
    ax.set_xticks(layers[::4])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.invert_yaxis()

plt.suptitle("Figure 2b: Mean Rank of Ground-Truth Token by Layer\n(Mistral-7B-Instruct-v0.2, 32 layers)",
             fontsize=12)
plt.tight_layout()
plt.savefig("figures/fig2b_gt_rank_mistral.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/fig2b_gt_rank_mistral.png")

print(f"\n{'Lang':<10} {'Category':<12} {'Mean rank @ layer 1':>20} {'Mean rank @ layer 32':>22}")
print("-" * 68)
for lang in LANGS:
    for cat in CATEGORIES:
        rm = results[lang][cat]["ranks"]
        print(f"{lang:<10} {cat:<12} {rm[:, 0].mean():>20.1f} {rm[:, -1].mean():>22.1f}")

# --- Figure 3b: Top-1 confidence by layer ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, cat in zip(axes, CATEGORIES):
    for lang, (color, marker) in lang_styles.items():
        tm = results[lang][cat]["top1"]
        ax.plot(layers, tm.mean(axis=0), marker=marker, color=color,
                label=lang, linewidth=1.8, markersize=4)
    ax.set_title(cat.capitalize(), fontsize=11)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean top-1 probability")
    ax.set_xticks(layers[::4])
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Figure 3b: Mean Top-1 Confidence by Layer\n(Mistral-7B-Instruct-v0.2, 32 layers)",
             fontsize=12)
plt.tight_layout()
plt.savefig("figures/fig3b_confidence_mistral.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/fig3b_confidence_mistral.png")

In [ ]:
# Cell 3 — RSA: Compute RDMs and Spearman correlation curves
from scipy.stats import spearmanr
from scipy.spatial.distance import cosine as cosine_distance

def compute_rdm(vecs_at_layer):
    """(n_q, d_model) → (n_q, n_q) cosine-distance RDM"""
    n = vecs_at_layer.shape[0]
    rdm = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            d = cosine_distance(vecs_at_layer[i], vecs_at_layer[j])
            rdm[i, j] = rdm[j, i] = d
    return rdm

def upper_triangle(m):
    return m[np.triu_indices(m.shape[0], k=1)]

LANG_PAIRS = [("English", "Spanish"), ("English", "Basque"), ("Spanish", "Basque")]
rsa_corr = {cat: {pair: [] for pair in LANG_PAIRS} for cat in CATEGORIES}

print("Computing RDMs and Spearman RSA correlations...")
for cat in CATEGORIES:
    for layer_idx in range(N_LAYERS):
        rdms = {lang: compute_rdm(rsa_vectors[lang][cat][:, layer_idx, :]) for lang in LANGS}
        for (la, lb) in LANG_PAIRS:
            r, _ = spearmanr(upper_triangle(rdms[la]), upper_triangle(rdms[lb]))
            rsa_corr[cat][(la, lb)].append(r)
    for pair in LANG_PAIRS:
        rsa_corr[cat][pair] = np.array(rsa_corr[cat][pair])
    print(f"  {cat}: done")

print("RSA computation complete.")

In [ ]:
# Cell 4 — Figure 4: RSA cross-lingual correlation by layer

pair_styles = {
    ("English", "Spanish"): ("royalblue",   "-",  "EN–ES"),
    ("English", "Basque"):  ("tomato",      "--", "EN–EU"),
    ("Spanish", "Basque"):  ("forestgreen", "-.", "ES–EU"),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, cat in zip(axes, CATEGORIES):
    for pair, (color, ls, label) in pair_styles.items():
        ax.plot(layers, rsa_corr[cat][pair], color=color, linestyle=ls,
                label=label, linewidth=2.0)
    ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
    ax.set_title(cat.capitalize(), fontsize=11)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Spearman r (cross-lingual RDM similarity)")
    ax.set_xticks(layers[::4])
    ax.set_ylim(-0.3, 1.0)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Figure 4: Cross-Lingual RSA Correlation by Layer\n"
    "(Mistral-7B-Instruct-v0.2, Spearman r between language-pair RDMs)",
    fontsize=12
)
plt.tight_layout()
plt.savefig("figures/fig4_rsa_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/fig4_rsa_correlation.png")

print(f"\n{'Category':<12} {'Pair':<14} {'Layer 1 r':>10} {'Layer 32 r':>11}")
print("-" * 52)
for cat in CATEGORIES:
    for pair, (_, _, label) in pair_styles.items():
        c = rsa_corr[cat][pair]
        print(f"{cat:<12} {label:<14} {c[0]:>10.3f} {c[-1]:>11.3f}")

In [ ]:
# Cell 5 — Figure 6: Side-by-side BLOOM-560M vs Mistral-7B-Instruct-v0.2
#
# Re-runs logit lens for BLOOM-560M locally (CPU, ~2min) for direct comparison.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

print("Loading BLOOM-560M for comparison (CPU)...")
small_tok   = AutoTokenizer.from_pretrained("bigscience/bloom-560m")
small_model = AutoModelForCausalLM.from_pretrained("bigscience/bloom-560m", torch_dtype=torch.float32)
small_model.eval()
n_layers_small = small_model.config.num_hidden_layers  # 24
print(f"BLOOM-560M: {n_layers_small} layers")

def load_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]

en_by_id = {d["id"]: d for d in load_jsonl("data/english.jsonl")}
es_by_id = {d["id"]: d for d in load_jsonl("data/spanish.jsonl")}
eu_by_id = {d["id"]: d for d in load_jsonl("data/basque.jsonl")}
LANGS_MAP = {"English": en_by_id, "Spanish": es_by_id, "Basque": eu_by_id}
common_ids = sorted(set(en_by_id) & set(es_by_id) & set(eu_by_id))

N_PER_CAT = 20
results_small = {lang: {cat: {"ranks": [], "top1": []} for cat in CATEGORIES} for lang in LANGS}
seen = {lang: {cat: 0 for cat in CATEGORIES} for lang in LANGS}
tasks_small = []
for qid in common_ids:
    cat = en_by_id[qid]["category"]
    if all(seen[lang][cat] < N_PER_CAT for lang in LANGS):
        for lang in LANGS:
            tasks_small.append((lang, LANGS_MAP[lang], qid, cat))
        for lang in LANGS:
            seen[lang][cat] += 1

print(f"Running BLOOM-560M logit lens on {len(tasks_small)} pairs...")
for i, (lang, store, qid, cat) in enumerate(tasks_small):
    item = store[qid]
    gt_ids = small_tok.encode(" " + item["ground_truth"].strip(), add_special_tokens=False)
    gt_id  = gt_ids[0] if gt_ids else None
    inputs = small_tok(item["question"], return_tensors="pt")
    with torch.no_grad():
        out = small_model(**inputs, output_hidden_states=True)
    ranks, top1 = [], []
    for h in out.hidden_states[1:]:
        h_last = h[0, -1, :]
        probs  = small_model.lm_head(small_model.transformer.ln_f(h_last)).softmax(dim=-1)
        top1.append(probs.max().item())
        ranks.append((probs.argsort(descending=True) == gt_id).nonzero(as_tuple=True)[0].item()
                     if gt_id is not None else -1)
    results_small[lang][cat]["ranks"].append(ranks)
    results_small[lang][cat]["top1"].append(top1)
    if (i + 1) % 30 == 0:
        print(f"  {i+1}/{len(tasks_small)}")

for lang in LANGS:
    for cat in CATEGORIES:
        results_small[lang][cat]["ranks"] = np.array(results_small[lang][cat]["ranks"])
        results_small[lang][cat]["top1"]  = np.array(results_small[lang][cat]["top1"])

layers_small = list(range(1, n_layers_small + 1))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, cat in enumerate(CATEGORIES):
    for lang, (color, marker) in lang_styles.items():
        axes[0, col].plot(layers_small,
                          results_small[lang][cat]["ranks"].mean(axis=0),
                          marker=marker, color=color, label=lang, linewidth=1.8, markersize=4)
        axes[1, col].plot(layers,
                          results[lang][cat]["ranks"].mean(axis=0),
                          marker=marker, color=color, label=lang, linewidth=1.8, markersize=4)
    for row in range(2):
        axes[row, col].set_title(cat.capitalize(), fontsize=11)
        axes[row, col].set_xlabel("Layer")
        axes[row, col].legend(fontsize=8)
        axes[row, col].grid(True, alpha=0.3)
        axes[row, col].invert_yaxis()

axes[0, 0].set_ylabel("BLOOM-560M\nMean GT rank")
axes[1, 0].set_ylabel("Mistral-7B-Instruct\nMean GT rank")
plt.suptitle("Figure 6: Logit Lens — BLOOM-560M (top) vs Mistral-7B-Instruct-v0.2 (bottom)",
             fontsize=12)
plt.tight_layout()
plt.savefig("figures/fig6_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/fig6_comparison.png")